# Reducers & Annotated State

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) LangGraph roadmap.

You will learn how LangGraph merges state updates using reducers, including `operator.add`, `add_messages`, and custom reducer functions.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Default Behavior — Overwrite

Without a reducer, returning a key replaces the previous value.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    value: str

def node_a(state: State) -> dict:
    return {"value": "A"}

def node_b(state: State) -> dict:
    return {"value": "B"}

graph = StateGraph(State)
graph.add_node("a", node_a)
graph.add_node("b", node_b)
graph.add_edge(START, "a")
graph.add_edge("a", "b")
graph.add_edge("b", END)

app = graph.compile()
result = app.invoke({"value": ""})
print(f"Final value: {result['value']}")

## 4. operator.add for List Appending

Use `operator.add` to concatenate lists instead of overwriting.

In [ ]:
import operator
from typing import Annotated

class ListState(TypedDict):
    steps: Annotated[list[str], operator.add]

def step_one(state: ListState) -> dict:
    return {"steps": ["step_one completed"]}

def step_two(state: ListState) -> dict:
    return {"steps": ["step_two completed"]}

graph = StateGraph(ListState)
graph.add_node("step_one", step_one)
graph.add_node("step_two", step_two)
graph.add_edge(START, "step_one")
graph.add_edge("step_one", "step_two")
graph.add_edge("step_two", END)

app = graph.compile()
result = app.invoke({"steps": []})
print(f"Steps: {result['steps']}")

## 5. Custom Reducer Function

Any function with signature `fn(current, new) -> merged` can serve as a reducer.

In [ ]:
def combine_logs(current: str, new: str) -> str:
    return f"{current} | {new}" if current else new

class LogState(TypedDict):
    log: Annotated[str, combine_logs]

def first(state: LogState) -> dict:
    return {"log": "first ran"}

def second(state: LogState) -> dict:
    return {"log": "second ran"}

graph = StateGraph(LogState)
graph.add_node("first", first)
graph.add_node("second", second)
graph.add_edge(START, "first")
graph.add_edge("first", "second")
graph.add_edge("second", END)

app = graph.compile()
result = app.invoke({"log": ""})
print(f"Log: {result['log']}")

## 6. add_messages Reducer

The `add_messages` reducer appends messages and deduplicates by ID.

In [ ]:
from langgraph.graph import add_messages
from langchain_core.messages import HumanMessage, AIMessage

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

messages = []
messages = add_messages(messages, [HumanMessage(content="Hello")])
messages = add_messages(messages, [AIMessage(content="Hi there!")])

for msg in messages:
    print(f"{msg.type}: {msg.content}")

## 7. MessagesState Shortcut

`MessagesState` is a prebuilt state with `messages: Annotated[list, add_messages]`.

In [ ]:
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

def chatbot(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(MessagesState)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

app = graph.compile()

result = app.invoke({"messages": [HumanMessage(content="What is a reducer in LangGraph?")]})
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}")

## What You Learned

1. **Default behavior** overwrites state fields on each node return
2. **`operator.add`** concatenates lists across nodes
3. **Custom reducers** let you define any merge logic with `Annotated[type, fn]`
4. **`add_messages`** is a purpose-built reducer for chat message lists with ID-based deduplication
5. **`MessagesState`** is a one-liner shortcut for the common messages-only state